# External Library Logging — `loguru`

Goal: use a third-party logging library that's simpler than the standard `logging` module. [`loguru`](https://github.com/Delgan/loguru) is the most popular choice for this — one import, no boilerplate, colored console output, and easy file logging with rotation, all out of the box.

Compare this to `_01_basic_python_logging.ipynb`: everything there (handlers, formatters, `getLogger`, levels, propagation) still exists conceptually — `loguru` just gives it sane defaults so you don't have to configure it yourself.

Install with: `pip install loguru`

## 1. Zero-config logging

**Why we need it:** with stdlib `logging`, getting readable console output takes a `getLogger` + `StreamHandler` + `Formatter` + `basicConfig` dance. `loguru` exposes one ready-to-use object — `logger` — that's already wired to the console with colors, timestamps, level, and source location.

In [1]:
from loguru import logger

# no setup needed — this already prints a colored, formatted line to the console
logger.debug("this prints — loguru's default level is DEBUG, unlike stdlib's WARNING")
logger.info("application started")
logger.warning("disk usage is high")
logger.error("failed to save file")
logger.critical("out of memory")

2026-08-16 12:28:30.305 | DEBUG    | __main__:<module>:4 - this prints — loguru's default level is DEBUG, unlike stdlib's WARNING


2026-08-16 12:28:30.305 | INFO     | __main__:<module>:5 - application started


2026-08-16 12:28:30.306 | WARNING  | __main__:<module>:6 - disk usage is high


2026-08-16 12:28:30.306 | ERROR    | __main__:<module>:7 - failed to save file


2026-08-16 12:28:30.306 | CRITICAL | __main__:<module>:8 - out of memory


## 2. Sinks — `logger.remove()` / `logger.add()`

**Why we need it:** in `loguru`, a "sink" is just a destination (console, file, function...) — the equivalent of a stdlib `Handler`. There's no separate `Formatter` class; format and level are just arguments to `add()`. `remove()` clears existing sinks (loguru starts with one default console sink) so you can replace it with your own config instead of stacking on top of it.

In [2]:
import sys

logger.remove()  # drop loguru's default console sink so we don't get duplicate lines below

logger.add(
    sys.stderr,
    level="INFO",
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan> | <level>{message}</level>",
)

logger.debug("this is now hidden — sink level is INFO")
logger.info("payment authorized for order #1042")
logger.warning("order #1042 has no shipping address")

12:28:30 | INFO     | __main__ | payment authorized for order #1042


12:28:30 | WARNING  | __main__ | order #1042 has no shipping address


## 3. Logging to a file, with automatic rotation

**Why we need it:** with stdlib `logging` you'd reach for `RotatingFileHandler` and configure size limits by hand. In `loguru`, a file sink with rotation is just another `add()` call — pass a path plus a `rotation` rule and it's handled for you (by size, time, or a custom condition).

In [3]:
logger.add(
    "logs/app.log",
    level="DEBUG",
    rotation="1 MB",       # start a new file once the current one hits 1 MB
    retention="3 days",    # delete rotated files older than this
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name} | {message}",
)

logger.debug("this goes to both the console sink (if level allowed) and the file sink")
logger.info("order #1042 shipped")

with open("logs/app.log") as f:
    print(f.read())

12:28:30 | INFO     | __main__ | order #1042 shipped


2026-08-16 12:28:30 | DEBUG    | __main__ | this goes to both the console sink (if level allowed) and the file sink
2026-08-16 12:28:30 | INFO     | __main__ | order #1042 shipped



## 4. Exceptions with full, readable tracebacks

**Why we need it:** like stdlib's `logger.exception(...)`, `loguru.logger.exception(...)` logs the current exception with its traceback. `loguru` goes further — it can show the actual *values* of variables at each frame, which is often the difference between guessing and knowing why something failed.

In [1]:
def compute_order_total(prices: list[float], quantity: int) -> float:
    return sum(prices) / quantity


try:
    compute_order_total(prices=[10.0, 20.0], quantity=0)
except ZeroDivisionError:
    logger.exception("failed to compute order total")

NameError: name 'logger' is not defined

## 5. Structured context with `bind()`

**Why we need it:** stitching request/user IDs into every message by hand (`f"[order={order_id}] ..."`) is repetitive and easy to forget. `bind()` returns a logger that carries extra key-value context on every call, so you can filter or format on it later without threading it through every log line manually.

In [5]:
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | order={extra[order_id]} | <level>{message}</level>",
)

order_logger = logger.bind(order_id=1042)
order_logger.info("payment authorized")
order_logger.info("shipment created")

12:28:30 | INFO     | order=1042 | payment authorized


12:28:30 | INFO     | order=1042 | shipment created


## Summary

- `from loguru import logger` gives you a ready-to-use, colored console logger — no `getLogger`/`Handler`/`Formatter`/`basicConfig` setup required.
- A **sink** (`logger.add(...)`) is loguru's equivalent of a stdlib handler — console, file, or any callable. `logger.remove()` clears sinks before reconfiguring.
- File sinks get rotation and retention (`rotation="1 MB"`, `retention="3 days"`) as plain arguments, no separate `RotatingFileHandler` class.
- `logger.exception(...)` logs a caught exception with a traceback — loguru's version can also show variable values per frame.
- `logger.bind(key=value)` attaches structured context to every message from the returned logger, without repeating it in each call.

Trade-off vs. the stdlib approach in `_01_basic_python_logging.ipynb`: `loguru` is faster to get productive with, but it's a third-party dependency — for a library meant to be imported by other people's applications, stdlib `logging` is still the safer default since it doesn't force your dependency on downstream consumers.